# A nearest-mean classifier without loops

A nearest-mean classifier stores one mean point per class and gives a new example the
class whose mean is closest. Here you build one with NumPy alone and no Python loop over the
examples: the class means as one matrix product, the predictions from a table of squared
distances, and then an honest estimate of its accuracy that never lets an example vote on
its own class mean.

**How to work through it.** Each part asks for one function. Replace the `TODO` and the
`raise NotImplementedError` with your code, then run the check cell under it. The check
prints `Part N passed` when your function is right, and stops with a message that says
what is wrong when it is not.

## Setup

Run this cell first.

In [ ]:
import numpy as np

# Seven seeds from three plant varieties: length, width and weight, already scaled.
X = np.array([[1.0, 2.0, 0.0],
              [3.0, 2.0, 2.0],
              [2.0, 5.0, 1.0],
              [0.0, 6.0, 3.0],
              [2.0, 8.0, 1.0],
              [7.0, 1.0, 5.0],
              [9.0, 3.0, 5.0]])
y = np.array([0, 0, 0, 1, 1, 2, 2])
k = 3
print(X.shape, np.bincount(y))

## Part 1 (coding task): The class means

Write `class_means(X, y, k)` for labels $0, \dots, k - 1$, where every class has at least
one example. It returns a $k \times d$ array whose row $c$ is the mean of the rows of `X`
with label $c$.

Do it without a loop: build the $n \times k$ one-hot matrix `O` with `O[i, c] = 1` when
`y[i] == c` (row `y[i]` of `np.eye(k)`), so that `O.T @ X` holds the sum of each class's
rows, then divide by the class counts.

In [ ]:
def class_means(X, y, k):
    """k x d array: row c is the mean of the rows of X labelled c."""
    # TODO
    raise NotImplementedError

In [ ]:
M = class_means(X, y, k)
print(M)
assert M.shape == (3, 3), f"expected shape (3, 3), got {M.shape}"
assert not np.allclose(M, (np.eye(3)[y].T @ X) / len(y)), "every class sum was divided by n: divide by that class's own count"
assert not np.allclose(M, (np.eye(3)[y].T @ X) / np.bincount(y)), "the counts were divided across the columns: give them shape (k, 1)"
assert np.allclose(M, [[2, 3, 1], [1, 7, 2], [8, 2, 5]]), "some class means are wrong"
rng = np.random.default_rng(0)
Xr, yr = rng.standard_normal((50, 2)), np.repeat(np.arange(4), [5, 10, 15, 20])
expected = np.array([Xr[yr == c].mean(axis=0) for c in range(4)])
assert np.allclose(class_means(Xr, yr, 4), expected), "wrong means on 50 random points in 4 classes of different sizes"
print("Part 1 passed")

## Part 2 (coding task): Predictions

Write `predict(X, means)`, which returns one label per row of `X`: the index of the closest
row of `means` in squared distance. Build the full $n \times k$ table of squared distances
with broadcasting, then pick the smallest entry of each row. When two means are equally
close, the lower label wins (this is what `argmin` does).

In [ ]:
def predict(X, means):
    """One label per row of X: the index of the nearest row of means."""
    # TODO
    raise NotImplementedError

In [ ]:
pred = predict(X, class_means(X, y, k))
print(pred)
assert pred.shape == (7,), f"expected one label per example, shape (7,), got {pred.shape}"
assert np.array_equal(pred, y), "on the seed data every seed is closest to its own class mean"
means = np.array([[0.0, 0.0], [3.0, 4.0], [10.0, 0.0]])
points = np.array([[1.0, 4.0], [2.0, -1.0], [5.0, 0.0], [6.0, 4.0]])
p = predict(points, means)
assert p[0] == 1, "point (1, 4) is closest to (3, 4): squared distance 4, against 17 to (0, 0)"
assert not np.array_equal(p, [1, 0, 0, 1]), "(5, 0) went to (0, 0): in squared distance it is 20 from (3, 4) and 25 from the other two means"
assert np.array_equal(p, [1, 0, 1, 1]), f"expected [1, 0, 1, 1], got {p}"
assert predict(np.array([[5.0, 0.0]]), np.array([[0.0, 0.0], [10.0, 0.0]]))[0] == 0, "a tie should go to the lower label"
print("Part 2 passed")

## Part 3 (coding task): Leave one out, still without a loop

Scoring the classifier on the points that built its means flatters it, because each point
pulled its own class mean toward itself. A fairer score classifies each point with means
computed without it. Only its own class mean changes: for a point $x_i$ of class $c$ with
class sum $S_c$ and count $N_c$, it becomes

$$
\frac{S_c - x_i}{N_c - 1}.
$$

Write `loo_predict(X, y, k)`, which returns the label each point gets when its own class
mean leaves it out (every class has at least two examples). Use no Python loop: take the
full distance table, then overwrite each point's entry for its own class.

In [ ]:
def loo_predict(X, y, k):
    """One label per row of X, each point classified with its own class mean computed without it."""
    # TODO
    raise NotImplementedError

In [ ]:
def loo_reference(X, y, k):
    labels = []
    for i in range(len(X)):
        keep = np.arange(len(X)) != i
        M = np.array([X[keep][y[keep] == c].mean(axis=0) for c in range(k)])
        labels.append(((X[i] - M) ** 2).sum(axis=1).argmin())
    return np.array(labels)

loo = loo_predict(X, y, k)
print(loo, "accuracy", (loo == y).mean())
assert loo.shape == (7,), f"expected shape (7,), got {loo.shape}"
assert not np.array_equal(loo, y), "these are the ordinary predictions: seed 2's own class mean must leave seed 2 out"
assert np.array_equal(loo, [0, 0, 1, 1, 1, 2, 2]), f"expected [0, 0, 1, 1, 1, 2, 2], got {loo}"
rng = np.random.default_rng(1)
Xr, yr = rng.standard_normal((60, 2)), rng.permutation(np.repeat(np.arange(4), 15))
assert np.array_equal(loo_predict(Xr, yr, 4), loo_reference(Xr, yr, 4)), "wrong labels on 60 random points: only each point's own class column should change"
print("Part 3 passed")

## Solution

### Part 1: The class means

In [ ]:
def class_means(X, y, k):
    """k x d array: row c is the mean of the rows of X labelled c."""
    O = np.eye(k)[y]                 # (n, k) one-hot rows
    sums = O.T @ X                   # (k, d): row c adds up class c's rows
    counts = O.sum(axis=0)           # (k,): examples per class
    return sums / counts[:, None]    # (k, d) / (k, 1): one count per row

**Why it works.** Row $i$ of `O` has a single 1, in column `y[i]`. So entry $(c, j)$ of
`O.T @ X` adds up feature $j$ over exactly the examples of class $c$, and `O.sum(axis=0)`
counts them. The counts have shape `(k,)`, and on the seed data $k = d = 3$, so dividing by
them directly would still run: it would divide column $c$ by the count of class $c$. Giving
them shape `(k, 1)` divides each row by its own class's count.

### Part 2: Predictions

In [ ]:
def predict(X, means):
    """One label per row of X: the index of the nearest row of means."""
    diff = X[:, None, :] - means[None, :, :]     # (n, 1, d) - (1, k, d) -> (n, k, d)
    D = (diff ** 2).sum(axis=2)                  # (n, k) squared distances
    return D.argmin(axis=1)                      # nearest mean for each row

**Why it works.** `X[:, None, :]` puts the examples down axis 0 and `means[None, :, :]`
puts the classes along axis 1, so the difference holds every example minus every mean.
Squaring and summing over the features leaves the $n \times k$ table, and `argmin(axis=1)`
takes the closest mean in each row. `argmin` returns the first position of the smallest
entry, so ties go to the lower label.

### Part 3: Leave one out, still without a loop

In [ ]:
def loo_predict(X, y, k):
    """One label per row of X, each point classified with its own class mean computed without it."""
    O = np.eye(k)[y]
    sums, counts = O.T @ X, O.sum(axis=0)
    means = sums / counts[:, None]
    D = ((X[:, None, :] - means[None, :, :]) ** 2).sum(axis=2)    # (n, k)
    own = (sums[y] - X) / (counts[y] - 1)[:, None]                 # (n, d): own mean without the point
    D[np.arange(len(X)), y] = ((X - own) ** 2).sum(axis=1)         # replace one entry per row
    return D.argmin(axis=1)

**Why it works.** `sums[y]` and `counts[y]` pick, for every point at once, its own class's
sum and count, so `own` holds all $n$ left-out means in one `(n, d)` array. Pairing
`np.arange(n)` with `y` writes one entry per row, the column of that row's own class, and
the other $k - 1$ distances stay as they were, since leaving a point out does not move the
other classes' means. On the seeds the score drops from 7 out of 7 to 6 out of 7: seed 2,
at $(2, 5, 1)$, is 4 from its class mean but 6 from class 1's, and without it class 0's
mean moves to $(2, 2, 1)$, at distance 9.